# 🚀 A/B Clients RCA Dashboard v8

**Full Features Version**

## Features:
- ✅ Fixed pandas groupby error
- 📊 Hourly graph with metric buttons
- 📈 Performance Trends with metric buttons + Bar/Line toggle
- 🚫 Exclude Landing Pages with LIKE pattern matching + Save
- 🎨 Light theme (Purple & White)

## Cell 1: Imports and Configuration

In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import webbrowser
import os
import sys
import warnings
import importlib
from pathlib import Path
from IPython.display import display, HTML

warnings.simplefilter('ignore')

# Load clients config
with open('config/clients_config.json', 'r') as f:
    CLIENTS_CONFIG = json.load(f)

# Add config folder to path and import queries
sys.path.insert(0, 'config')
if 'queries' in sys.modules:
    del sys.modules['queries']
import queries
importlib.reload(queries)
from queries import QUERIES

# Load ClickHouse credentials
with open('credentials/clickhouse_secret.json', 'r') as f:
    CLICKHOUSE_SECRETS = json.load(f)

AVAILABLE_CLIENTS = list(CLIENTS_CONFIG.keys())

print("✅ Configuration loaded")
print(f"   Available clients: {', '.join(AVAILABLE_CLIENTS)}")

## Cell 2: Database Connection

In [ ]:
# ============================================================================
# CELL 2: DATABASE CONNECTION
# ============================================================================

import clickhouse_connect

client = clickhouse_connect.get_client(**CLICKHOUSE_SECRETS)
print("✅ ClickHouse connected successfully")

## Cell 3: Helper Functions

In [ ]:
# ============================================================================
# CELL 3: HELPER FUNCTIONS
# ============================================================================

def normalize_path(path):
    if pd.isna(path) or path == '':
        return '/'
    path = str(path).strip().rstrip('/')
    return path if path else '/'

def get_query(client_key, query_name):
    return QUERIES[client_key][query_name]

def check_client_data_available(client_key):
    csv_path = CLIENTS_CONFIG[client_key]['sessions_csv']
    return os.path.exists(csv_path)

CLIENTS_WITH_DATA = []
print("📊 Checking clients data availability...")
for c in AVAILABLE_CLIENTS:
    if check_client_data_available(c):
        CLIENTS_WITH_DATA.append(c)
        print(f"   {c}: ✅")
    else:
        print(f"   {c}: ❌ (missing sessions.csv)")

print(f"\n✅ Helper functions loaded")
print(f"📊 Clients ready: {', '.join(CLIENTS_WITH_DATA)}")

## Cell 4: Load All Client Data

In [ ]:
# ============================================================================
# CELL 4: LOAD ALL CLIENT DATA
# ============================================================================

ALL_CLIENT_DATA = {}

for CLIENT_KEY in CLIENTS_WITH_DATA:
    CLIENT_CONFIG = CLIENTS_CONFIG[CLIENT_KEY]
    print(f"\n{'='*60}")
    print(f"📊 Loading data for: {CLIENT_CONFIG['display_name']}")
    print(f"{'='*60}")
    
    try:
        # SHOPIFY SESSIONS (from CSV)
        sessions_df = pd.read_csv(CLIENT_CONFIG['sessions_csv'])
        sessions_df['UTM source'] = sessions_df['UTM source'].fillna('')
        sessions_df['UTM campaign'] = sessions_df['UTM campaign'].fillna('')
        if sessions_df['Bounce rate'].max() <= 1:
            sessions_df['Bounce rate'] = sessions_df['Bounce rate'] * 100
        sessions_df['Day'] = pd.to_datetime(sessions_df['Day'])
        sessions_df['Landing page path'] = sessions_df['Landing page path'].apply(normalize_path)
        print(f"   ✅ Sessions CSV: {len(sessions_df):,} rows")
        
        # SHOPIFY ORDERS BY DAY
        query = get_query(CLIENT_KEY, 'shopify_orders_by_day')
        result = client.query(query)
        orders_daily = pd.DataFrame(result.result_set, columns=['Day', 'Orders', 'Revenue'])
        orders_daily['Day'] = pd.to_datetime(orders_daily['Day'])
        print(f"   ✅ Orders by day: {orders_daily['Orders'].sum():,} orders")
        
        # SHOPIFY ORDERS BY LANDING PAGE
        query = get_query(CLIENT_KEY, 'shopify_orders_by_lp')
        result = client.query(query)
        orders_by_lp = pd.DataFrame(result.result_set, columns=['Day', 'Landing page path', 'Orders', 'Revenue'])
        orders_by_lp['Day'] = pd.to_datetime(orders_by_lp['Day'])
        orders_by_lp['Landing page path'] = orders_by_lp['Landing page path'].apply(normalize_path)
        print(f"   ✅ Orders by LP: {len(orders_by_lp):,} rows")
        
        # SHOPIFY ORDERS BY UTM
        query = get_query(CLIENT_KEY, 'shopify_orders_by_utm')
        result = client.query(query)
        orders_by_utm = pd.DataFrame(result.result_set, columns=['Day', 'UTM source', 'Orders', 'Revenue'])
        orders_by_utm['Day'] = pd.to_datetime(orders_by_utm['Day'])
        print(f"   ✅ Orders by UTM: {len(orders_by_utm):,} rows")
        
        # SHOPIFY ORDERS BY HOUR
        query = get_query(CLIENT_KEY, 'shopify_orders_by_hour')
        result = client.query(query)
        orders_by_hour = pd.DataFrame(result.result_set, columns=['Day', 'Hour of day', 'Orders', 'Revenue'])
        orders_by_hour['Day'] = pd.to_datetime(orders_by_hour['Day'])
        print(f"   ✅ Orders by hour: {len(orders_by_hour):,} rows")
        
        # TECTONIC DATA
        query = get_query(CLIENT_KEY, 'tectonic_data')
        result = client.query(query)
        tectonic_df = pd.DataFrame(result.result_set, columns=result.column_names)
        tectonic_df['Day'] = pd.to_datetime(tectonic_df['Day'])
        print(f"   ✅ Tectonic: {tectonic_df['sessions'].sum():,} sessions | {tectonic_df['orders'].sum():,.0f} orders")
        
        ALL_CLIENT_DATA[CLIENT_KEY] = {
            'config': CLIENT_CONFIG,
            'sessions': sessions_df,
            'orders_daily': orders_daily,
            'orders_by_lp': orders_by_lp,
            'orders_by_utm': orders_by_utm,
            'orders_by_hour': orders_by_hour,
            'tectonic': tectonic_df
        }
    except Exception as e:
        print(f"   ❌ Error loading {CLIENT_KEY}: {e}")

print(f"\n{'='*60}")
print(f"✅ Data loaded for {len(ALL_CLIENT_DATA)} client(s): {', '.join(ALL_CLIENT_DATA.keys())}")

## Cell 5: Date Filters

In [ ]:
# ============================================================================
# CELL 5: DATE FILTERS
# ============================================================================

def get_date_filters():
    today = pd.Timestamp.now(tz='Asia/Kolkata').date()
    yesterday = today - timedelta(days=1)
    seven_days_ago = today - timedelta(days=7)
    thirty_days_ago = today - timedelta(days=30)
    
    return {
        'today': {'func': lambda d: d.dt.date == today, 'label': f'Today ({today})'},
        'yesterday': {'func': lambda d: d.dt.date == yesterday, 'label': f'Yesterday ({yesterday})'},
        'last_7_days': {'func': lambda d: (d.dt.date >= seven_days_ago) & (d.dt.date <= yesterday), 'label': f'Last 7 Days'},
        'last_30_days': {'func': lambda d: (d.dt.date >= thirty_days_ago) & (d.dt.date <= yesterday), 'label': f'Last 30 Days'}
    }

date_filters = get_date_filters()
print("✅ Date filters created")

## Cell 6: Analysis Functions (FIXED for pandas)

In [ ]:
# ============================================================================
# CELL 6: ANALYSIS FUNCTIONS (FIXED for newer pandas)
# ============================================================================
# FIX: Use for-loop groupby instead of groupby().apply().reset_index()
# ============================================================================

import pandas as pd
import numpy as np

def normalize_path_for_matching(path, max_length=200):
    if pd.isna(path) or path == '':
        return '/'
    path = str(path).strip().rstrip('/')
    if '?' in path:
        path = path.split('?')[0]
    if not path.startswith('/'):
        path = '/' + path
    if len(path) > max_length:
        path = path[:max_length]
    return path.lower()

def calculate_summary(shopify_sessions, shopify_orders_daily, tectonic_df, filter_func):
    s_sess = shopify_sessions[filter_func(shopify_sessions['Day'])].copy()
    s_orders = shopify_orders_daily[filter_func(shopify_orders_daily['Day'])].copy()
    t_df = tectonic_df[filter_func(tectonic_df['Day'])].copy()
    
    s_sessions = s_sess['Sessions'].sum()
    s_atc = s_sess['Sessions with cart additions'].sum() if 'Sessions with cart additions' in s_sess.columns else 0
    s_checkout = s_sess['Sessions that reached checkout'].sum() if 'Sessions that reached checkout' in s_sess.columns else 0
    s_bounce = (s_sess['Bounce rate'] * s_sess['Sessions']).sum() / s_sessions if s_sessions > 0 else 0
    s_orders_total = s_orders['Orders'].sum()
    s_revenue = s_orders['Revenue'].sum()
    
    t_sessions = t_df['sessions'].sum()
    t_atc = t_df['atc_sessions'].sum()
    t_checkout = t_df['checkout_sessions'].sum()
    t_bounce = (t_df['bounce_sessions'].sum() / t_sessions * 100) if t_sessions > 0 else 0
    t_orders = t_df['orders'].sum()
    t_revenue = t_df['revenue'].sum()
    
    total_sessions = s_sessions + t_sessions
    
    results = []
    for label, sess, atc, checkout, orders, rev, bounce in [
        ('Shopify', s_sessions, s_atc, s_checkout, s_orders_total, s_revenue, s_bounce),
        ('Tectonic', t_sessions, t_atc, t_checkout, t_orders, t_revenue, t_bounce)
    ]:
        atc_rate = (atc / sess * 100) if sess > 0 else 0
        conv = (orders / sess * 100) if sess > 0 else 0
        rps = rev / sess if sess > 0 else 0
        aov = rev / orders if orders > 0 else 0
        traffic = (sess / total_sessions * 100) if total_sessions > 0 else 0
        
        results.append({
            'source': label, 'sessions': sess, 'traffic_split': traffic,
            'bounce_rate': bounce, 'atc_rate': atc_rate, 'conversion': conv,
            'orders': orders, 'revenue': rev, 'rps': rps, 'aov': aov
        })
    
    s, t = results[0], results[1]
    delta = {
        'sessions_delta': t['sessions'] - s['sessions'],
        'bounce_rate_delta': t['bounce_rate'] - s['bounce_rate'],
        'atc_rate_delta': t['atc_rate'] - s['atc_rate'],
        'conversion_delta': t['conversion'] - s['conversion'],
        'orders_delta': t['orders'] - s['orders'],
        'revenue_delta': t['revenue'] - s['revenue'],
        'rps_delta': t['rps'] - s['rps'],
        'aov_delta': t['aov'] - s['aov']
    }
    
    return results, delta

def analyze_landing_pages(shopify_sessions, shopify_orders_by_lp, tectonic_df, filter_func, top_n=30):
    s_sess = shopify_sessions[filter_func(shopify_sessions['Day'])].copy()
    s_orders = shopify_orders_by_lp[filter_func(shopify_orders_by_lp['Day'])].copy()
    t_df = tectonic_df[filter_func(tectonic_df['Day'])].copy()
    
    s_sess['Landing page path'] = s_sess['Landing page path'].apply(normalize_path_for_matching)
    s_orders['Landing page path'] = s_orders['Landing page path'].apply(normalize_path_for_matching)
    t_df['Landing page path'] = t_df['Landing page path'].apply(normalize_path_for_matching)
    
    # FIXED: Use for-loop instead of groupby().apply().reset_index()
    s_lp_list = []
    for lp, group in s_sess.groupby('Landing page path'):
        s_lp_list.append({
            'Landing page path': lp,
            'Sessions': group['Sessions'].sum(),
            'ATC Sessions': group['Sessions with cart additions'].sum() if 'Sessions with cart additions' in group.columns else 0,
            'Bounce Rate': (group['Bounce rate'] * group['Sessions']).sum() / group['Sessions'].sum() if group['Sessions'].sum() > 0 else 0
        })
    s_lp_agg = pd.DataFrame(s_lp_list) if s_lp_list else pd.DataFrame(columns=['Landing page path', 'Sessions', 'ATC Sessions', 'Bounce Rate'])
    
    s_lp_orders = s_orders.groupby('Landing page path').agg({'Orders': 'sum', 'Revenue': 'sum'}).reset_index()
    shopify_lp = s_lp_agg.merge(s_lp_orders, on='Landing page path', how='left').fillna(0)
    
    shopify_lp['ATC Rate'] = (shopify_lp['ATC Sessions'] / shopify_lp['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_lp['Conversion'] = (shopify_lp['Orders'] / shopify_lp['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_lp['RPS'] = (shopify_lp['Revenue'] / shopify_lp['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_lp['AOV'] = (shopify_lp['Revenue'] / shopify_lp['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    tectonic_lp = t_df.groupby('Landing page path').agg({
        'sessions': 'sum', 'bounce_sessions': 'sum', 'atc_sessions': 'sum', 'orders': 'sum', 'revenue': 'sum'
    }).reset_index()
    tectonic_lp.columns = ['Landing page path', 'Sessions', 'Bounce Sessions', 'ATC Sessions', 'Orders', 'Revenue']
    
    tectonic_lp['Bounce Rate'] = (tectonic_lp['Bounce Sessions'] / tectonic_lp['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_lp['ATC Rate'] = (tectonic_lp['ATC Sessions'] / tectonic_lp['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_lp['Conversion'] = (tectonic_lp['Orders'] / tectonic_lp['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_lp['RPS'] = (tectonic_lp['Revenue'] / tectonic_lp['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_lp['AOV'] = (tectonic_lp['Revenue'] / tectonic_lp['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    comparison = shopify_lp.merge(tectonic_lp, on='Landing page path', how='outer', suffixes=('_shopify', '_tectonic')).fillna(0)
    comparison['Total Sessions'] = comparison['Sessions_shopify'] + comparison['Sessions_tectonic']
    
    def pct_delta(t_val, s_val):
        if s_val == 0: return 0 if t_val == 0 else 100
        return ((t_val - s_val) / s_val) * 100
    
    comparison['Delta Bounce Rate'] = comparison.apply(lambda r: pct_delta(r['Bounce Rate_tectonic'], r['Bounce Rate_shopify']), axis=1)
    comparison['Delta ATC Rate'] = comparison.apply(lambda r: pct_delta(r['ATC Rate_tectonic'], r['ATC Rate_shopify']), axis=1)
    comparison['Delta Conversion'] = comparison.apply(lambda r: pct_delta(r['Conversion_tectonic'], r['Conversion_shopify']), axis=1)
    comparison['Delta RPS'] = comparison.apply(lambda r: pct_delta(r['RPS_tectonic'], r['RPS_shopify']), axis=1)
    comparison['Delta AOV'] = comparison.apply(lambda r: pct_delta(r['AOV_tectonic'], r['AOV_shopify']), axis=1)
    
    totals = {
        'Sessions_shopify': comparison['Sessions_shopify'].sum(),
        'Sessions_tectonic': comparison['Sessions_tectonic'].sum(),
        'Orders_shopify': comparison['Orders_shopify'].sum(),
        'Orders_tectonic': comparison['Orders_tectonic'].sum(),
        'Revenue_shopify': comparison['Revenue_shopify'].sum(),
        'Revenue_tectonic': comparison['Revenue_tectonic'].sum(),
    }
    
    numeric_cols = comparison.select_dtypes(include=[np.number]).columns
    comparison[numeric_cols] = comparison[numeric_cols].round(2)
    
    return {'records': comparison.nlargest(top_n, 'Total Sessions').to_dict('records'), 'totals': totals}

def analyze_utm_sources(shopify_sessions, shopify_orders_by_utm, tectonic_df, filter_func, top_n=20):
    s_sess = shopify_sessions[filter_func(shopify_sessions['Day'])].copy()
    s_orders = shopify_orders_by_utm[filter_func(shopify_orders_by_utm['Day'])].copy()
    t_df = tectonic_df[filter_func(tectonic_df['Day'])].copy()
    
    # FIXED: Use for-loop
    s_utm_list = []
    for utm, group in s_sess.groupby('UTM source'):
        s_utm_list.append({
            'UTM source': utm, 'Sessions': group['Sessions'].sum(),
            'ATC Sessions': group['Sessions with cart additions'].sum() if 'Sessions with cart additions' in group.columns else 0,
            'Bounce Rate': (group['Bounce rate'] * group['Sessions']).sum() / group['Sessions'].sum() if group['Sessions'].sum() > 0 else 0
        })
    s_utm_agg = pd.DataFrame(s_utm_list) if s_utm_list else pd.DataFrame(columns=['UTM source', 'Sessions', 'ATC Sessions', 'Bounce Rate'])
    
    s_utm_orders = s_orders.groupby('UTM source').agg({'Orders': 'sum', 'Revenue': 'sum'}).reset_index()
    shopify_utm = s_utm_agg.merge(s_utm_orders, on='UTM source', how='left').fillna(0)
    
    shopify_utm['ATC Rate'] = (shopify_utm['ATC Sessions'] / shopify_utm['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_utm['Conversion'] = (shopify_utm['Orders'] / shopify_utm['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_utm['RPS'] = (shopify_utm['Revenue'] / shopify_utm['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_utm['AOV'] = (shopify_utm['Revenue'] / shopify_utm['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    tectonic_utm = t_df.groupby('UTM source').agg({'sessions': 'sum', 'bounce_sessions': 'sum', 'atc_sessions': 'sum', 'orders': 'sum', 'revenue': 'sum'}).reset_index()
    tectonic_utm.columns = ['UTM source', 'Sessions', 'Bounce Sessions', 'ATC Sessions', 'Orders', 'Revenue']
    
    tectonic_utm['Bounce Rate'] = (tectonic_utm['Bounce Sessions'] / tectonic_utm['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_utm['ATC Rate'] = (tectonic_utm['ATC Sessions'] / tectonic_utm['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_utm['Conversion'] = (tectonic_utm['Orders'] / tectonic_utm['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_utm['RPS'] = (tectonic_utm['Revenue'] / tectonic_utm['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_utm['AOV'] = (tectonic_utm['Revenue'] / tectonic_utm['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    comparison = shopify_utm.merge(tectonic_utm, on='UTM source', how='outer', suffixes=('_shopify', '_tectonic')).fillna(0)
    comparison['Total Sessions'] = comparison['Sessions_shopify'] + comparison['Sessions_tectonic']
    
    def pct_delta(t_val, s_val):
        if s_val == 0: return 0 if t_val == 0 else 100
        return ((t_val - s_val) / s_val) * 100
    
    comparison['Delta Bounce Rate'] = comparison.apply(lambda r: pct_delta(r['Bounce Rate_tectonic'], r['Bounce Rate_shopify']), axis=1)
    comparison['Delta ATC Rate'] = comparison.apply(lambda r: pct_delta(r['ATC Rate_tectonic'], r['ATC Rate_shopify']), axis=1)
    comparison['Delta Conversion'] = comparison.apply(lambda r: pct_delta(r['Conversion_tectonic'], r['Conversion_shopify']), axis=1)
    comparison['Delta RPS'] = comparison.apply(lambda r: pct_delta(r['RPS_tectonic'], r['RPS_shopify']), axis=1)
    comparison['Delta AOV'] = comparison.apply(lambda r: pct_delta(r['AOV_tectonic'], r['AOV_shopify']), axis=1)
    
    totals = {
        'Sessions_shopify': comparison['Sessions_shopify'].sum(),
        'Sessions_tectonic': comparison['Sessions_tectonic'].sum(),
        'Orders_shopify': comparison['Orders_shopify'].sum(),
        'Orders_tectonic': comparison['Orders_tectonic'].sum(),
    }
    
    numeric_cols = comparison.select_dtypes(include=[np.number]).columns
    comparison[numeric_cols] = comparison[numeric_cols].round(2)
    
    return {'records': comparison.nlargest(top_n, 'Total Sessions').to_dict('records'), 'totals': totals}

def analyze_hourly(shopify_sessions, shopify_orders_by_hour, tectonic_df, filter_func):
    s_sess = shopify_sessions[filter_func(shopify_sessions['Day'])].copy()
    s_orders = shopify_orders_by_hour[filter_func(shopify_orders_by_hour['Day'])].copy()
    t_df = tectonic_df[filter_func(tectonic_df['Day'])].copy()
    
    # FIXED: Use for-loop
    s_hourly_list = []
    for hour, group in s_sess.groupby('Hour of day'):
        s_hourly_list.append({
            'Hour of day': hour, 'Sessions': group['Sessions'].sum(),
            'ATC Sessions': group['Sessions with cart additions'].sum() if 'Sessions with cart additions' in group.columns else 0,
            'Bounce Rate': (group['Bounce rate'] * group['Sessions']).sum() / group['Sessions'].sum() if group['Sessions'].sum() > 0 else 0
        })
    s_hourly_agg = pd.DataFrame(s_hourly_list) if s_hourly_list else pd.DataFrame(columns=['Hour of day', 'Sessions', 'ATC Sessions', 'Bounce Rate'])
    
    s_hourly_orders = s_orders.groupby('Hour of day').agg({'Orders': 'sum', 'Revenue': 'sum'}).reset_index()
    shopify_hourly = s_hourly_agg.merge(s_hourly_orders, on='Hour of day', how='left').fillna(0)
    
    shopify_hourly['ATC Rate'] = (shopify_hourly['ATC Sessions'] / shopify_hourly['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_hourly['Conversion'] = (shopify_hourly['Orders'] / shopify_hourly['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_hourly['RPS'] = (shopify_hourly['Revenue'] / shopify_hourly['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_hourly['AOV'] = (shopify_hourly['Revenue'] / shopify_hourly['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    tectonic_hourly = t_df.groupby('Hour of day').agg({'sessions': 'sum', 'bounce_sessions': 'sum', 'atc_sessions': 'sum', 'orders': 'sum', 'revenue': 'sum'}).reset_index()
    tectonic_hourly.columns = ['Hour of day', 'Sessions', 'Bounce Sessions', 'ATC Sessions', 'Orders', 'Revenue']
    
    tectonic_hourly['Bounce Rate'] = (tectonic_hourly['Bounce Sessions'] / tectonic_hourly['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_hourly['ATC Rate'] = (tectonic_hourly['ATC Sessions'] / tectonic_hourly['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_hourly['Conversion'] = (tectonic_hourly['Orders'] / tectonic_hourly['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_hourly['RPS'] = (tectonic_hourly['Revenue'] / tectonic_hourly['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_hourly['AOV'] = (tectonic_hourly['Revenue'] / tectonic_hourly['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    all_hours = pd.DataFrame({'Hour of day': range(24)})
    comparison = all_hours.merge(shopify_hourly, on='Hour of day', how='left').fillna(0)
    comparison = comparison.merge(tectonic_hourly, on='Hour of day', how='left', suffixes=('_shopify', '_tectonic')).fillna(0)
    
    def pct_delta(t_val, s_val):
        if s_val == 0: return 0 if t_val == 0 else 100
        return ((t_val - s_val) / s_val) * 100
    
    comparison['Delta Bounce Rate'] = comparison.apply(lambda r: pct_delta(r['Bounce Rate_tectonic'], r['Bounce Rate_shopify']), axis=1)
    comparison['Delta ATC Rate'] = comparison.apply(lambda r: pct_delta(r['ATC Rate_tectonic'], r['ATC Rate_shopify']), axis=1)
    comparison['Delta Conversion'] = comparison.apply(lambda r: pct_delta(r['Conversion_tectonic'], r['Conversion_shopify']), axis=1)
    comparison['Delta RPS'] = comparison.apply(lambda r: pct_delta(r['RPS_tectonic'], r['RPS_shopify']), axis=1)
    comparison['Delta AOV'] = comparison.apply(lambda r: pct_delta(r['AOV_tectonic'], r['AOV_shopify']), axis=1)
    
    numeric_cols = comparison.select_dtypes(include=[np.number]).columns
    comparison[numeric_cols] = comparison[numeric_cols].round(2)
    
    return comparison.sort_values('Hour of day').to_dict('records')

def calculate_daily_trends(shopify_sessions, shopify_orders_daily, tectonic_df, filter_func):
    s_sess = shopify_sessions[filter_func(shopify_sessions['Day'])].copy()
    s_orders = shopify_orders_daily[filter_func(shopify_orders_daily['Day'])].copy()
    t_df = tectonic_df[filter_func(tectonic_df['Day'])].copy()
    
    # FIXED: Use for-loop
    s_daily_list = []
    for day, group in s_sess.groupby('Day'):
        s_daily_list.append({
            'Day': day, 'Sessions': group['Sessions'].sum(),
            'ATC Sessions': group['Sessions with cart additions'].sum() if 'Sessions with cart additions' in group.columns else 0,
            'Bounce Rate': (group['Bounce rate'] * group['Sessions']).sum() / group['Sessions'].sum() if group['Sessions'].sum() > 0 else 0
        })
    s_daily_sess = pd.DataFrame(s_daily_list) if s_daily_list else pd.DataFrame(columns=['Day', 'Sessions', 'ATC Sessions', 'Bounce Rate'])
    
    s_daily_orders = s_orders.groupby('Day').agg({'Orders': 'sum', 'Revenue': 'sum'}).reset_index()
    shopify_daily = s_daily_sess.merge(s_daily_orders, on='Day', how='outer').fillna(0)
    
    shopify_daily['ATC Rate'] = (shopify_daily['ATC Sessions'] / shopify_daily['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_daily['Conversion'] = (shopify_daily['Orders'] / shopify_daily['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_daily['RPS'] = (shopify_daily['Revenue'] / shopify_daily['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    shopify_daily['AOV'] = (shopify_daily['Revenue'] / shopify_daily['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    tectonic_daily = t_df.groupby('Day').agg({'sessions': 'sum', 'bounce_sessions': 'sum', 'atc_sessions': 'sum', 'orders': 'sum', 'revenue': 'sum'}).reset_index()
    tectonic_daily.columns = ['Day', 'Sessions', 'Bounce Sessions', 'ATC Sessions', 'Orders', 'Revenue']
    tectonic_daily['Bounce Rate'] = (tectonic_daily['Bounce Sessions'] / tectonic_daily['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_daily['ATC Rate'] = (tectonic_daily['ATC Sessions'] / tectonic_daily['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_daily['Conversion'] = (tectonic_daily['Orders'] / tectonic_daily['Sessions'] * 100).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_daily['RPS'] = (tectonic_daily['Revenue'] / tectonic_daily['Sessions']).replace([float('inf'), -float('inf')], 0).fillna(0)
    tectonic_daily['AOV'] = (tectonic_daily['Revenue'] / tectonic_daily['Orders']).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    all_dates = pd.DataFrame({'Day': pd.concat([shopify_daily['Day'], tectonic_daily['Day']]).unique()}).sort_values('Day')
    comparison = all_dates.merge(shopify_daily, on='Day', how='left', suffixes=('', '_s')).fillna(0)
    comparison = comparison.merge(tectonic_daily, on='Day', how='left', suffixes=('_shopify', '_tectonic')).fillna(0)
    comparison['Date'] = pd.to_datetime(comparison['Day']).dt.strftime('%Y-%m-%d')
    
    numeric_cols = comparison.select_dtypes(include=[np.number]).columns
    comparison[numeric_cols] = comparison[numeric_cols].round(2)
    
    return comparison.sort_values('Day').to_dict('records')

print("✅ Analysis functions loaded (FIXED for newer pandas)")

## Cell 7: Generate Analysis for All Clients

In [ ]:
# ============================================================================
# CELL 7: GENERATE ANALYSIS FOR ALL CLIENTS
# ============================================================================

ALL_DASHBOARD_DATA = {}

for CLIENT_KEY, client_data in ALL_CLIENT_DATA.items():
    print(f"\n📊 Generating analysis for {client_data['config']['display_name']}...")
    
    client_dashboard = {}
    
    for period_key, period_info in date_filters.items():
        filter_func = period_info['func']
        
        summary, delta = calculate_summary(
            client_data['sessions'], client_data['orders_daily'], client_data['tectonic'], filter_func
        )
        landing_pages = analyze_landing_pages(
            client_data['sessions'], client_data['orders_by_lp'], client_data['tectonic'], filter_func
        )
        utm_sources = analyze_utm_sources(
            client_data['sessions'], client_data['orders_by_utm'], client_data['tectonic'], filter_func
        )
        hourly = analyze_hourly(
            client_data['sessions'], client_data['orders_by_hour'], client_data['tectonic'], filter_func
        )
        daily_trends = calculate_daily_trends(
            client_data['sessions'], client_data['orders_daily'], client_data['tectonic'], filter_func
        )
        
        client_dashboard[period_key] = {
            'summary': summary, 
            'delta': delta, 
            'landing_pages': landing_pages,
            'utm_sources': utm_sources, 
            'hourly': hourly,
            'daily_trends': daily_trends
        }
    
    ALL_DASHBOARD_DATA[CLIENT_KEY] = {
        'display_name': client_data['config']['display_name'],
        'currency': client_data['config'].get('currency', '₹'),
        'data': client_dashboard
    }
    print(f"   ✅ Done!")

print(f"\n{'='*60}")
print(f"✅ Analysis complete for {len(ALL_DASHBOARD_DATA)} client(s)!")

## Cell 8: HTML Dashboard Generator (WITH ALL FEATURES)

In [ ]:
#rca v8 dashbaord 
import json
from datetime import datetime

def generate_html_dashboard(all_client_data):
    client_options = '\n'.join([f'<option value="{key}">{data["display_name"]}</option>' for key, data in all_client_data.items()])
    tectonic_logo = '<svg style="height:22px;width:22px;vertical-align:middle;margin-right:6px;" viewBox="280 47 268 268" xmlns="http://www.w3.org/2000/svg"><path fill="#fff" d="M413.48,311.57c-9.89,0-19.76-1.11-29.32-3.31-23.89-5.48-45.73-17.58-63.15-35-18.31-18.31-30.67-41.34-35.74-66.62-1.7-8.47-2.56-17.16-2.56-25.85,0-18.58,3.83-36.56,11.37-53.4,6.53-14.59,15.59-27.73,26.94-39.07,17.41-17.43,39.24-29.53,63.14-35.01,9.56-2.2,19.43-3.31,29.33-3.31,14.97,0,29.67,2.51,43.67,7.47,18.21,6.45,35.09,17.11,48.81,30.83,11.34,11.34,20.4,24.49,26.94,39.07,7.54,16.84,11.37,34.82,11.37,53.4,0,8.69-.86,17.4-2.56,25.85-5.07,25.27-17.43,48.31-35.74,66.62-13.72,13.72-30.59,24.38-48.81,30.83-14,4.96-28.7,7.47-43.67,7.47h-.02Z"/><path fill="#000" d="M333.32,260.95c14.79,14.79,33.63,25.5,54.73,30.34,8.18,1.87,16.69,2.86,25.44,2.86,13.27,0,26.01-2.29,37.86-6.47,16.08-5.69,30.5-14.92,42.31-26.73,15.48-15.48,26.51-35.43,30.99-57.74,1.46-7.25,2.22-14.75,2.22-22.42,0-16.48-3.52-32.15-9.85-46.28-5.69-12.71-13.64-24.16-23.36-33.88-11.82-11.82-26.23-21.03-42.31-26.73-11.84-4.19-24.58-6.47-37.86-6.47-8.75,0-17.26.99-25.44,2.86-21.1,4.83-39.94,15.56-54.73,30.34-9.72,9.72-17.67,21.17-23.36,33.88-6.33,14.13-9.85,29.8-9.85,46.28,0,7.68.76,15.18,2.22,22.42,4.48,22.31,15.5,42.26,30.99,57.74Z"/><path fill="#fff" d="M395.58,201.1c-38.83-11.48-81.92-26.18-59.12-67.87,16.09-28.2,46.36-47.23,80.96-47.23,51.42,0,93.26,42.05,93.26,93.74,0,30.67-15.8,66.24-37.46,75.06-52.81,18.1-38.02-42-77.63-53.7h-.01Z"/><path fill="#000" d="M403.43,197.05c-29.93-8.8-63.15-20.07-45.58-52.05,12.4-21.63,35.73-36.23,62.41-36.23,39.64,0,71.9,32.25,71.9,71.89,0,23.52-12.18,50.8-28.88,57.57-40.71,13.88-29.31-32.2-59.84-41.19h-.01Z"/><path fill="#fff" d="M412.35,193.81c-19.5-5.73-41.14-13.08-29.69-33.9,8.08-14.09,23.27-23.6,40.65-23.6,25.82,0,46.83,21.01,46.83,46.83,0,15.32-7.93,33.1-18.81,37.5-26.52,9.04-19.09-20.98-38.98-26.83h.01Z"/><path fill="#000" d="M420.24,189.51c-10.68-3.15-22.54-7.17-16.27-18.58,4.43-7.72,12.75-12.94,22.28-12.94,14.15,0,25.66,11.51,25.66,25.67,0,8.4-4.35,18.14-10.31,20.55-14.54,4.96-10.47-11.5-21.36-14.70h-.01Z"/></svg>'
    generated_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    data_json = json.dumps(all_client_data, default=str)
    
    html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>A/B Clients RCA Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif; background: #e8eef5; color: #333; line-height: 1.6; }}
        .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 30px; text-align: center; box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4); }}
        .header h1 {{ font-size: 2.5rem; margin-bottom: 10px; text-shadow: 2px 2px 4px rgba(0,0,0,0.2); }}
        .container {{ max-width: 1800px; margin: 0 auto; padding: 20px; }}
        .controls {{ background: linear-gradient(145deg, #f0f4f8, #d9e2ec); padding: 20px 25px; border-radius: 16px; margin-bottom: 20px; box-shadow: 8px 8px 16px #c8d0d8, -8px -8px 16px #ffffff; display: flex; gap: 25px; align-items: center; flex-wrap: wrap; }}
        .controls label {{ font-weight: 600; color: #555; }}
        .controls select {{ padding: 12px 20px; font-size: 1rem; border: none; border-radius: 12px; background: linear-gradient(145deg, #ffffff, #e6e9ef); box-shadow: inset 2px 2px 5px #d1d5db, inset -2px -2px 5px #ffffff, 4px 4px 8px #c8d0d8, -4px -4px 8px #ffffff; cursor: pointer; min-width: 180px; }}
        .generated-time {{ margin-left: auto; color: #666; font-size: 0.85rem; background: rgba(255,255,255,0.5); padding: 8px 15px; border-radius: 20px; }}
        .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px; }}
        .metric-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }}
        .metric {{ text-align: center; padding: 15px 10px; background: linear-gradient(145deg, #f8fafc, #e8eef5); border-radius: 12px; box-shadow: 4px 4px 8px #d1d9e6, -4px -4px 8px #ffffff; transition: all 0.3s ease; }}
        .metric:hover {{ transform: translateY(-3px); box-shadow: 6px 6px 12px #c8d0d8, -6px -6px 12px #ffffff; }}
        .metric .value {{ font-size: 1.4rem; font-weight: 700; color: #333; }}
        .metric .label {{ font-size: 0.7rem; color: #666; margin-top: 5px; }}
        .metric.positive .value {{ color: #10b981; }}
        .metric.negative .value {{ color: #ef4444; }}
        .card {{ background: linear-gradient(145deg, #f5f8fc, #e4eaf3); border-radius: 20px; padding: 25px; box-shadow: 10px 10px 20px #d1d9e6, -10px -10px 20px #ffffff; position: relative; overflow: hidden; transition: all 0.3s ease; }}
        .card::before {{ content: ''; position: absolute; top: 0; left: 0; right: 0; height: 4px; background: linear-gradient(90deg, #667eea, #764ba2); opacity: 0; transition: opacity 0.3s; }}
        .card:hover::before {{ opacity: 1; }}
        .card h3 {{ color: #333; margin-bottom: 18px; font-size: 1.1rem; }}
        .chart-container {{ position: relative; height: 300px; margin-top: 20px; }}
        .full-width {{ grid-column: 1 / -1; }}
        .insight-box {{ background: linear-gradient(135deg, #f0f4f8 0%, #e4eaf3 100%); border-left: 4px solid #667eea; padding: 15px 20px; border-radius: 0 12px 12px 0; margin-top: 15px; }}
        .insight-item {{ padding: 6px 0; border-bottom: 1px dashed #d1d9e6; }}
        .insight-item:last-child {{ border-bottom: none; }}
        .tabs {{ display: flex; gap: 12px; margin-bottom: 20px; flex-wrap: wrap; }}
        .tab-btn {{ padding: 14px 28px; border: none; background: linear-gradient(145deg, #f0f4f8, #dce4ed); border-radius: 12px; cursor: pointer; font-weight: 600; box-shadow: 5px 5px 10px #c8d0d8, -5px -5px 10px #ffffff; color: #555; transition: all 0.3s ease; }}
        .tab-btn:hover {{ transform: translateY(-2px); }}
        .tab-btn.active {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; box-shadow: 3px 3px 8px #c8d0d8; }}
        .tab-content {{ display: none; }}
        .tab-content.active {{ display: block; }}
        .sort-btn, .page-btn {{ padding: 8px 16px; border: none; background: linear-gradient(145deg, #f0f4f8, #dce4ed); border-radius: 8px; cursor: pointer; font-weight: 600; margin: 0 3px; box-shadow: 3px 3px 6px #c8d0d8, -3px -3px 6px #ffffff; color: #555; font-size: 0.85rem; transition: all 0.2s ease; }}
        .sort-btn:hover, .page-btn:hover {{ transform: translateY(-1px); }}
        .sort-btn.active, .page-btn.active {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }}
        .chart-toggle {{ display: flex; gap: 5px; margin-left: 15px; }}
        .chart-toggle-btn {{ padding: 6px 12px; border: none; background: linear-gradient(145deg, #f0f4f8, #dce4ed); border-radius: 6px; cursor: pointer; font-size: 0.8rem; color: #666; box-shadow: 2px 2px 4px #c8d0d8, -2px -2px 4px #ffffff; transition: all 0.2s; }}
        .chart-toggle-btn:hover {{ transform: translateY(-1px); }}
        .chart-toggle-btn.active {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }}
        .lp-table-wrapper {{ position: relative; overflow: hidden; }}
        .lp-table-scroll {{ overflow-x: auto; margin-left: 280px; }}
        .lp-table {{ border-collapse: collapse; width: max-content; min-width: 100%; }}
        .lp-table th, .lp-table td {{ padding: 10px 12px; text-align: right; border-bottom: 1px solid #e0e6ed; white-space: nowrap; font-size: 0.85rem; }}
        .lp-table th {{ background: linear-gradient(145deg, #e8eef5, #dce4ed); font-weight: 600; color: #555; }}
        .lp-table th:first-child, .lp-table td:first-child {{ position: absolute; left: 0; width: 280px; background: #f5f8fc; text-align: left; box-shadow: 2px 0 5px rgba(0,0,0,0.1); }}
        .lp-table th:first-child {{ background: linear-gradient(145deg, #e8eef5, #dce4ed); z-index: 2; }}
        .lp-table tr:hover td {{ background: rgba(102, 126, 234, 0.05); }}
        .col-group-shopify {{ background: rgba(150, 191, 72, 0.15) !important; }}
        .col-group-tectonic {{ background: rgba(102, 126, 234, 0.15) !important; }}
        .col-group-delta {{ background: rgba(236, 72, 153, 0.1) !important; }}
        .delta-positive {{ color: #10b981; font-weight: 600; }}
        .delta-negative {{ color: #ef4444; font-weight: 600; }}
        .delta-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }}
        .delta-card {{ text-align: center; padding: 12px 8px; background: linear-gradient(145deg, #f8fafc, #e8eef5); border-radius: 12px; box-shadow: 4px 4px 8px #d1d9e6, -4px -4px 8px #ffffff; transition: all 0.3s ease; }}
        .delta-card:hover {{ transform: translateY(-3px); box-shadow: 6px 6px 12px #c8d0d8, -6px -6px 12px #ffffff; }}
        .delta-card .pct-value {{ font-size: 1.3rem; font-weight: 700; margin-bottom: 4px; }}
        .delta-card.positive .pct-value {{ color: #10b981; }}
        .delta-card.negative .pct-value {{ color: #ef4444; }}
        .delta-card .metric-label {{ font-size: 0.85rem; font-weight: 600; display: flex; align-items: center; justify-content: center; gap: 4px; color: #555; }}
        .delta-card .pts-value {{ font-size: 0.75rem; color: #888; font-weight: 500; }}
        .delta-card.positive .arrow {{ color: #10b981; }}
        .delta-card.negative .arrow {{ color: #ef4444; }}
        .totals-row {{ background: linear-gradient(145deg, #f0f4f8, #e4eaf3); padding: 15px 20px; border-radius: 12px; margin-top: 15px; box-shadow: 4px 4px 8px #d1d9e6, -4px -4px 8px #ffffff; }}
        .totals-row h4 {{ color: #667eea; margin-bottom: 10px; }}
        .totals-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(120px, 1fr)); gap: 15px; }}
        .total-item {{ text-align: center; }}
        .total-item .value {{ font-size: 1.1rem; font-weight: 700; color: #333; }}
        .total-item .label {{ font-size: 0.7rem; color: #666; }}
        
        /* Executive Summary Notebook Styles */
        .summary-notebook {{ background: linear-gradient(145deg, #fffef5, #f8f6e8); border-radius: 16px; padding: 25px; margin-top: 20px; box-shadow: 8px 8px 16px #d1d9e6, -8px -8px 16px #ffffff; border-left: 5px solid #667eea; position: relative; }}
        .summary-notebook::before {{ content: ''; position: absolute; top: 20px; bottom: 20px; left: 30px; width: 1px; background: repeating-linear-gradient(to bottom, #e0d5c5 0px, #e0d5c5 1px, transparent 1px, transparent 25px); }}
        .summary-notebook h4 {{ color: #667eea; font-size: 1.2rem; margin-bottom: 20px; padding-left: 40px; display: flex; align-items: center; gap: 10px; }}
        .notebook-content {{ padding-left: 40px; }}
        .notebook-section {{ margin-bottom: 20px; padding: 15px; background: rgba(255,255,255,0.6); border-radius: 10px; border: 1px solid #e8e0d0; }}
        .notebook-section h5 {{ color: #764ba2; font-size: 0.95rem; margin-bottom: 12px; display: flex; align-items: center; gap: 8px; }}
        .notebook-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 12px; }}
        .notebook-item {{ text-align: center; padding: 10px; background: linear-gradient(145deg, #ffffff, #f5f5f0); border-radius: 8px; box-shadow: 2px 2px 4px rgba(0,0,0,0.05); }}
        .notebook-item .nb-label {{ font-size: 0.7rem; color: #888; text-transform: uppercase; letter-spacing: 0.5px; }}
        .notebook-item .nb-value {{ font-size: 1.1rem; font-weight: 700; color: #333; margin-top: 4px; }}
        .notebook-item .nb-value.positive {{ color: #10b981; }}
        .notebook-item .nb-value.negative {{ color: #ef4444; }}
        .notebook-item .nb-value.shopify {{ color: #96bf48; }}
        .notebook-item .nb-value.tectonic {{ color: #667eea; }}
        .notebook-divider {{ height: 1px; background: linear-gradient(to right, transparent, #d1c4a8, transparent); margin: 15px 0; }}
        .verdict-box {{ background: linear-gradient(135deg, #667eea15, #764ba215); border-radius: 12px; padding: 15px 20px; margin-top: 15px; border: 1px solid #667eea30; }}
        .verdict-box .verdict-title {{ font-weight: 700; color: #667eea; margin-bottom: 8px; display: flex; align-items: center; gap: 8px; }}
        .verdict-box .verdict-text {{ font-size: 0.9rem; color: #555; line-height: 1.6; }}
        .performance-badge {{ display: inline-block; padding: 4px 12px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; }}
        .performance-badge.winning {{ background: #10b98120; color: #10b981; }}
        .performance-badge.losing {{ background: #ef444420; color: #ef4444; }}
        .performance-badge.neutral {{ background: #f59e0b20; color: #f59e0b; }}
        
        .exclude-btn {{ padding: 10px 20px; background: linear-gradient(145deg, #f0f4f8, #dce4ed); border: none; border-radius: 10px; color: #555; cursor: pointer; font-weight: 600; box-shadow: 3px 3px 6px #c8d0d8, -3px -3px 6px #ffffff; transition: all 0.2s ease; }}
        .exclude-btn:hover {{ transform: translateY(-1px); }}
        .exclude-count {{ background: #667eea; color: white; padding: 2px 8px; border-radius: 10px; font-size: 0.8rem; margin-left: 8px; }}
        .modal-overlay {{ display: none; position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: rgba(0,0,0,0.5); z-index: 1000; justify-content: center; align-items: center; }}
        .modal-overlay.active {{ display: flex; }}
        .modal {{ background: linear-gradient(145deg, #f5f8fc, #e4eaf3); border-radius: 20px; padding: 30px; width: 90%; max-width: 600px; max-height: 80vh; overflow-y: auto; box-shadow: 20px 20px 40px #c8d0d8; position: relative; }}
        .modal h2 {{ color: #667eea; margin-bottom: 20px; }}
        .modal-close {{ position: absolute; top: 15px; right: 20px; font-size: 1.5rem; cursor: pointer; color: #666; }}
        .exclude-input {{ flex: 1; padding: 12px 15px; border: none; border-radius: 10px; font-size: 1rem; background: linear-gradient(145deg, #ffffff, #e6e9ef); box-shadow: inset 2px 2px 5px #d1d5db; }}
        .btn {{ padding: 12px 24px; border: none; border-radius: 10px; cursor: pointer; font-weight: 600; transition: all 0.2s ease; }}
        .btn:hover {{ transform: translateY(-1px); }}
        .btn-primary {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }}
        .btn-secondary {{ background: linear-gradient(145deg, #f0f4f8, #dce4ed); color: #555; }}
        .btn-danger {{ background: #ef4444; color: white; }}
        .excluded-list {{ margin-top: 20px; max-height: 200px; overflow-y: auto; }}
        .excluded-item {{ display: flex; justify-content: space-between; align-items: center; padding: 10px 15px; background: linear-gradient(145deg, #ffffff, #f0f4f8); border-radius: 8px; margin-bottom: 8px; box-shadow: 2px 2px 5px #d1d9e6; }}
        .excluded-item code {{ font-family: monospace; color: #667eea; background: #e8eef5; padding: 2px 8px; border-radius: 4px; }}
        .excluded-item .remove-btn {{ background: none; border: none; color: #ef4444; cursor: pointer; font-size: 1.2rem; }}
        .save-indicator {{ display: none; padding: 10px 20px; background: #10b981; color: white; border-radius: 8px; margin-top: 15px; text-align: center; }}
        .save-indicator.show {{ display: block; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🚀 A/B Clients RCA Dashboard</h1>
        <p>Shopify vs Tectonic Performance Analysis - <span id="current-client"></span></p>
    </div>
    <div class="container">
        <div class="controls">
            <div><label>Client:</label><select id="client-select" onchange="updateDashboard()">{client_options}</select></div>
            <div><label>Time Period:</label><select id="period-select" onchange="updateDashboard()"><option value="today">Today</option><option value="yesterday" selected>Yesterday</option><option value="last_7_days">Last 7 Days</option><option value="last_30_days">Last 30 Days</option></select></div>
            <button class="exclude-btn" onclick="openExcludeModal()">🚫 Exclude LPs<span class="exclude-count" id="exclude-count">0</span></button>
            <div class="generated-time">Generated: {generated_time}</div>
        </div>
        <div class="tabs">
            <button class="tab-btn active" onclick="showTab('summary')">📊 Summary</button>
            <button class="tab-btn" onclick="showTab('landing-pages')">🔗 Landing Pages</button>
            <button class="tab-btn" onclick="showTab('utm-sources')">📢 UTM Sources</button>
            <button class="tab-btn" onclick="showTab('hourly')">⏰ Hourly</button>
        </div>
        <div id="summary-tab" class="tab-content active">
            <div class="grid">
                <div class="card"><h3>🛍️ Shopify Performance</h3><div class="metric-grid" id="shopify-metrics"></div></div>
                <div class="card"><h3>{tectonic_logo} Tectonic Performance</h3><div class="metric-grid" id="tectonic-metrics"></div></div>
                <div class="card full-width"><h3>📈 Delta Analysis (Tectonic vs Shopify)</h3><div class="delta-grid" id="delta-metrics"></div><div class="insight-box" id="insight-box"></div></div>
            </div>
            <div class="card full-width">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px; flex-wrap: wrap; gap: 10px;">
                    <h3 style="margin: 0;">📊 Performance Trends</h3>
                    <div style="display: flex; align-items: center; flex-wrap: wrap; gap: 10px;">
                        <div>
                            <button class="sort-btn active" onclick="changeTrendMetric('Sessions', this)">Sessions</button>
                            <button class="sort-btn" onclick="changeTrendMetric('Bounce Rate', this)">Bounce%</button>
                            <button class="sort-btn" onclick="changeTrendMetric('ATC Rate', this)">ATC%</button>
                            <button class="sort-btn" onclick="changeTrendMetric('Conversion', this)">Conv%</button>
                            <button class="sort-btn" onclick="changeTrendMetric('Orders', this)">Orders</button>
                            <button class="sort-btn" onclick="changeTrendMetric('Revenue', this)">Revenue</button>
                            <button class="sort-btn" onclick="changeTrendMetric('RPS', this)">RPS</button>
                            <button class="sort-btn" onclick="changeTrendMetric('AOV', this)">AOV</button>
                        </div>
                        <div class="chart-toggle">
                            <button class="chart-toggle-btn active" onclick="setTrendChartType('bar', this)">📊 Bar</button>
                            <button class="chart-toggle-btn" onclick="setTrendChartType('line', this)">📈 Line</button>
                        </div>
                    </div>
                </div>
                <div class="chart-container" style="height: 350px;"><canvas id="trends-chart"></canvas></div>
                <div id="trends-totals" class="totals-row"></div>
            </div>
            <div id="executive-summary" class="summary-notebook"></div>
        </div>
        <div id="landing-pages-tab" class="tab-content">
            <div class="card">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;">
                    <h3 style="margin: 0;">🔗 Top Landing Pages</h3>
                    <div><button class="sort-btn active" onclick="sortLP('Total Sessions', this)">Total</button><button class="sort-btn" onclick="sortLP('Sessions_shopify', this)">Shopify</button><button class="sort-btn" onclick="sortLP('Sessions_tectonic', this)">Tectonic</button></div>
                </div>
                <div class="lp-table-wrapper"><div class="lp-table-scroll"><table class="lp-table" id="lp-table"><thead><tr><th>Landing Page</th><th class="col-group-shopify">S:Sess</th><th class="col-group-shopify">S:Bounce%</th><th class="col-group-shopify">S:ATC%</th><th class="col-group-shopify">S:Conv%</th><th class="col-group-shopify">S:Orders</th><th class="col-group-shopify">S:Revenue</th><th class="col-group-shopify">S:RPS</th><th class="col-group-shopify">S:AOV</th><th class="col-group-tectonic">T:Sess</th><th class="col-group-tectonic">T:Bounce%</th><th class="col-group-tectonic">T:ATC%</th><th class="col-group-tectonic">T:Conv%</th><th class="col-group-tectonic">T:Orders</th><th class="col-group-tectonic">T:Revenue</th><th class="col-group-tectonic">T:RPS</th><th class="col-group-tectonic">T:AOV</th><th class="col-group-delta">⚡Bounce</th><th class="col-group-delta">⚡ATC</th><th class="col-group-delta">⚡Conv</th><th class="col-group-delta">⚡RPS</th><th class="col-group-delta">⚡AOV</th></tr></thead><tbody></tbody></table></div></div>
                <div id="lp-pagination" style="display: flex; justify-content: center; gap: 5px; margin-top: 15px;"></div>
            </div>
        </div>
        <div id="utm-sources-tab" class="tab-content">
            <div class="card">
                <h3>📢 Top UTM Sources</h3>
                <table class="lp-table" id="utm-table" style="width:100%;"><thead><tr><th style="position:relative;width:auto;">UTM Source</th><th class="col-group-shopify">S:Sess</th><th class="col-group-shopify">S:Conv%</th><th class="col-group-shopify">S:Orders</th><th class="col-group-shopify">S:Revenue</th><th class="col-group-tectonic">T:Sess</th><th class="col-group-tectonic">T:Conv%</th><th class="col-group-tectonic">T:Orders</th><th class="col-group-tectonic">T:Revenue</th><th class="col-group-delta">⚡Conv</th><th class="col-group-delta">⚡RPS</th></tr></thead><tbody></tbody></table>
            </div>
        </div>
        <div id="hourly-tab" class="tab-content">
            <div class="card">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px; flex-wrap: wrap; gap: 10px;">
                    <h3 style="margin: 0;">⏰ Hourly Performance</h3>
                    <div>
                        <button class="sort-btn active" onclick="changeHourlyMetric('Sessions', this)">Sessions</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('Bounce Rate', this)">Bounce%</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('ATC Rate', this)">ATC%</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('Conversion', this)">Conv%</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('Orders', this)">Orders</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('Revenue', this)">Revenue</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('RPS', this)">RPS</button>
                        <button class="sort-btn" onclick="changeHourlyMetric('AOV', this)">AOV</button>
                    </div>
                </div>
                <div class="chart-container" style="height: 350px;"><canvas id="hourly-chart"></canvas></div>
                <div id="hourly-totals" class="totals-row"></div>
            </div>
        </div>
    </div>
    <div class="modal-overlay" id="exclude-modal">
        <div class="modal">
            <span class="modal-close" onclick="closeExcludeModal()">&times;</span>
            <h2>🚫 Exclude Landing Pages</h2>
            <p style="color: #666; margin-bottom: 20px;">Add patterns to exclude (e.g., <code>/pages</code> excludes all URLs containing "/pages")</p>
            <div style="display: flex; gap: 10px; margin-bottom: 15px;">
                <input type="text" class="exclude-input" id="exclude-pattern" placeholder="/pages, /blog, /cart..." onkeypress="if(event.key==='Enter') addExcludePattern()">
                <button class="btn btn-primary" onclick="addExcludePattern()">+ Add</button>
            </div>
            <div class="excluded-list" id="excluded-list"></div>
            <div style="display: flex; gap: 10px; margin-top: 20px;">
                <button class="btn btn-primary" onclick="saveExclusions()">💾 Save & Apply</button>
                <button class="btn btn-secondary" onclick="closeExcludeModal()">Close</button>
                <button class="btn btn-danger" onclick="clearAllExclusions()" style="margin-left: auto;">Clear All</button>
            </div>
            <div class="save-indicator" id="save-indicator">✅ Saved!</div>
        </div>
    </div>
    <script>
        const allData = {data_json};
        let trendsChart = null, hourlyChart = null;
        let currentLpSort = 'Total Sessions', currentLpPage = 1;
        let currentHourlyMetric = 'Sessions';
        let currentTrendMetric = 'Sessions';
        let currentTrendChartType = 'bar';
        const lpPerPage = 15;
        let excludedPatterns = [];
        let summaryData = {{}};
        
        function loadExclusions() {{ const k = 'rca_exclude_' + document.getElementById('client-select').value; excludedPatterns = JSON.parse(localStorage.getItem(k) || '[]'); updateExcludeCount(); renderExcludedList(); }}
        function saveExclusions() {{ const k = 'rca_exclude_' + document.getElementById('client-select').value; localStorage.setItem(k, JSON.stringify(excludedPatterns)); document.getElementById('save-indicator').classList.add('show'); setTimeout(() => document.getElementById('save-indicator').classList.remove('show'), 2000); updateDashboard(); }}
        function addExcludePattern() {{ const i = document.getElementById('exclude-pattern'); const p = i.value.trim(); if (p && !excludedPatterns.includes(p)) {{ excludedPatterns.push(p); i.value = ''; updateExcludeCount(); renderExcludedList(); }} }}
        function removeExcludePattern(p) {{ excludedPatterns = excludedPatterns.filter(x => x !== p); updateExcludeCount(); renderExcludedList(); }}
        function clearAllExclusions() {{ if (confirm('Clear all?')) {{ excludedPatterns = []; updateExcludeCount(); renderExcludedList(); }} }}
        function updateExcludeCount() {{ document.getElementById('exclude-count').textContent = excludedPatterns.length; }}
        function renderExcludedList() {{ document.getElementById('excluded-list').innerHTML = excludedPatterns.length === 0 ? '<p style="color:#999;text-align:center;">No patterns</p>' : excludedPatterns.map(p => `<div class="excluded-item"><code>${{p}}</code><button class="remove-btn" onclick="removeExcludePattern('${{p}}')">&times;</button></div>`).join(''); }}
        function isExcluded(lp) {{ const path = (lp['Landing page path'] || '').toLowerCase(); return excludedPatterns.some(p => path.includes(p.toLowerCase())); }}
        function openExcludeModal() {{ document.getElementById('exclude-modal').classList.add('active'); }}
        function closeExcludeModal() {{ document.getElementById('exclude-modal').classList.remove('active'); }}
        function getClientData() {{ return allData[document.getElementById('client-select').value]; }}
        function formatNumber(n) {{ return new Intl.NumberFormat('en-IN').format(Math.round(n)); }}
        function formatDecimal(n) {{ return n.toFixed(2); }}
        function formatCurrency(n, c) {{ return c + new Intl.NumberFormat('en-IN').format(n.toFixed(2)); }}
        function showTab(id) {{ document.querySelectorAll('.tab-content').forEach(t => t.classList.remove('active')); document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active')); document.getElementById(id + '-tab').classList.add('active'); event.target.classList.add('active'); if (id === 'hourly') setTimeout(updateHourlyChart, 100); if (id === 'summary') setTimeout(updateTrendsChart, 100); }}
        function setTrendChartType(type, btn) {{ currentTrendChartType = type; document.querySelectorAll('.chart-toggle-btn').forEach(b => b.classList.remove('active')); btn.classList.add('active'); updateTrendsChart(); }}
        function changeTrendMetric(metric, btn) {{ currentTrendMetric = metric; document.querySelectorAll('#summary-tab .sort-btn').forEach(b => b.classList.remove('active')); btn.classList.add('active'); updateTrendsChart(); }}
        function changeHourlyMetric(metric, btn) {{ currentHourlyMetric = metric; document.querySelectorAll('#hourly-tab .sort-btn').forEach(b => b.classList.remove('active')); btn.classList.add('active'); updateHourlyChart(); }}
        
        function updateDashboard() {{
            loadExclusions();
            const clientData = getClientData();
            const period = document.getElementById('period-select').value;
            const periodData = clientData.data[period];
            const currency = clientData.currency;
            document.getElementById('current-client').textContent = clientData.display_name;
            if (!periodData) return;
            const lpRecords = periodData.landing_pages.records || [];
            const filteredLP = lpRecords.filter(lp => !isExcluded(lp));
            let s_sessions = 0, s_orders = 0, s_revenue = 0, s_atc = 0, s_bounce_sum = 0;
            let t_sessions = 0, t_orders = 0, t_revenue = 0, t_atc = 0, t_bounce_sum = 0;
            filteredLP.forEach(lp => {{
                s_sessions += lp.Sessions_shopify || 0; s_orders += lp.Orders_shopify || 0; s_revenue += lp.Revenue_shopify || 0;
                s_atc += lp['ATC Sessions_shopify'] || 0; s_bounce_sum += (lp['Bounce Rate_shopify'] || 0) * (lp.Sessions_shopify || 0);
                t_sessions += lp.Sessions_tectonic || 0; t_orders += lp.Orders_tectonic || 0; t_revenue += lp.Revenue_tectonic || 0;
                t_atc += lp['ATC Sessions_tectonic'] || 0; t_bounce_sum += (lp['Bounce Rate_tectonic'] || 0) * (lp.Sessions_tectonic || 0);
            }});
            const total = s_sessions + t_sessions;
            const s = {{ sessions: s_sessions, traffic_split: total > 0 ? s_sessions/total*100 : 0, bounce_rate: s_sessions > 0 ? s_bounce_sum/s_sessions : 0, atc_rate: s_sessions > 0 ? s_atc/s_sessions*100 : 0, conversion: s_sessions > 0 ? s_orders/s_sessions*100 : 0, orders: s_orders, revenue: s_revenue, rps: s_sessions > 0 ? s_revenue/s_sessions : 0, aov: s_orders > 0 ? s_revenue/s_orders : 0 }};
            const t = {{ sessions: t_sessions, traffic_split: total > 0 ? t_sessions/total*100 : 0, bounce_rate: t_sessions > 0 ? t_bounce_sum/t_sessions : 0, atc_rate: t_sessions > 0 ? t_atc/t_sessions*100 : 0, conversion: t_sessions > 0 ? t_orders/t_sessions*100 : 0, orders: t_orders, revenue: t_revenue, rps: t_sessions > 0 ? t_revenue/t_sessions : 0, aov: t_orders > 0 ? t_revenue/t_orders : 0 }};
            summaryData = {{ s, t, currency, period, clientName: clientData.display_name }};
            updateSummary(s, t, currency);
            updateTrendsChart();
            updateLPTable(filteredLP, currency);
            updateUTMTable(periodData.utm_sources, currency);
            updateHourlyChart();
            updateExecutiveSummary(s, t, currency);
        }}
        
        function updateSummary(s, t, currency) {{
            document.getElementById('shopify-metrics').innerHTML = `<div class="metric"><div class="value">${{formatNumber(s.sessions)}}</div><div class="label">Sessions</div></div><div class="metric"><div class="value">${{s.traffic_split.toFixed(1)}}%</div><div class="label">Traffic Split</div></div><div class="metric"><div class="value">${{s.bounce_rate.toFixed(2)}}%</div><div class="label">Bounce Rate</div></div><div class="metric"><div class="value">${{s.atc_rate.toFixed(2)}}%</div><div class="label">ATC Rate</div></div><div class="metric"><div class="value">${{s.conversion.toFixed(2)}}%</div><div class="label">Conversion</div></div><div class="metric"><div class="value">${{formatNumber(s.orders)}}</div><div class="label">Orders</div></div><div class="metric"><div class="value">${{formatCurrency(s.revenue, currency)}}</div><div class="label">Revenue</div></div><div class="metric"><div class="value">${{formatCurrency(s.rps, currency)}}</div><div class="label">RPS</div></div><div class="metric"><div class="value">${{formatCurrency(s.aov, currency)}}</div><div class="label">AOV</div></div>`;
            document.getElementById('tectonic-metrics').innerHTML = `<div class="metric"><div class="value">${{formatNumber(t.sessions)}}</div><div class="label">Sessions</div></div><div class="metric"><div class="value">${{t.traffic_split.toFixed(1)}}%</div><div class="label">Traffic Split</div></div><div class="metric"><div class="value">${{t.bounce_rate.toFixed(2)}}%</div><div class="label">Bounce Rate</div></div><div class="metric"><div class="value">${{t.atc_rate.toFixed(2)}}%</div><div class="label">ATC Rate</div></div><div class="metric"><div class="value">${{t.conversion.toFixed(2)}}%</div><div class="label">Conversion</div></div><div class="metric"><div class="value">${{formatNumber(t.orders)}}</div><div class="label">Orders</div></div><div class="metric"><div class="value">${{formatCurrency(t.revenue, currency)}}</div><div class="label">Revenue</div></div><div class="metric"><div class="value">${{formatCurrency(t.rps, currency)}}</div><div class="label">RPS</div></div><div class="metric"><div class="value">${{formatCurrency(t.aov, currency)}}</div><div class="label">AOV</div></div>`;
            
            const calcPct = (tVal, sVal) => sVal === 0 ? 0 : ((tVal - sVal) / sVal) * 100;
            
            const createDeltaCard = (name, tVal, sVal, isCurrency = false) => {{
                const pctChange = calcPct(tVal, sVal);
                const ptsDelta = tVal - sVal;
                const isPositive = ptsDelta >= 0;
                const cls = isPositive ? 'positive' : 'negative';
                const arrow = isPositive ? '↑' : '↓';
                const pctDisplay = (pctChange >= 0 ? '+' : '') + pctChange.toFixed(2) + '%';
                const ptsDisplay = isCurrency 
                    ? ((ptsDelta >= 0 ? '+' : '') + currency + Math.abs(ptsDelta).toFixed(2))
                    : ((ptsDelta >= 0 ? '+' : '') + ptsDelta.toFixed(2) + ' pts');
                
                return `<div class="delta-card ${{cls}}">
                    <div class="pct-value">${{pctDisplay}}</div>
                    <div class="metric-label"><span class="arrow">${{arrow}}</span> ${{name}} <span class="pts-value">(${{ptsDisplay}})</span></div>
                </div>`;
            }};
            
            document.getElementById('delta-metrics').innerHTML = 
                createDeltaCard('ATC Rate', t.atc_rate, s.atc_rate) +
                createDeltaCard('Conversion', t.conversion, s.conversion) +
                createDeltaCard('RPS', t.rps, s.rps, true) +
                createDeltaCard('AOV', t.aov, s.aov, true);
            
            // All 4 metrics insights
            const atcDelta = t.atc_rate - s.atc_rate;
            const convDelta = t.conversion - s.conversion;
            const rpsDelta = t.rps - s.rps;
            const aovDelta = t.aov - s.aov;
            
            const getInsight = (name, delta, pct, isCurrency, currSymbol) => {{
                const icon = delta >= 0 ? '✅' : '⚠️';
                const dir = delta >= 0 ? 'higher' : 'lower';
                const valStr = isCurrency ? currSymbol + Math.abs(delta).toFixed(2) : Math.abs(delta).toFixed(2) + ' pts';
                return `<div class="insight-item">${{icon}} <strong>${{name}}:</strong> Tectonic is ${{Math.abs(pct).toFixed(2)}}% ${{dir}} (${{valStr}})</div>`;
            }};
            
            document.getElementById('insight-box').innerHTML = 
                getInsight('ATC Rate', atcDelta, calcPct(t.atc_rate, s.atc_rate), false, '') +
                getInsight('Conversion', convDelta, calcPct(t.conversion, s.conversion), false, '') +
                getInsight('RPS', rpsDelta, calcPct(t.rps, s.rps), true, currency) +
                getInsight('AOV', aovDelta, calcPct(t.aov, s.aov), true, currency);
        }}
        
        function updateExecutiveSummary(s, t, currency) {{
            const calcPct = (tVal, sVal) => sVal === 0 ? 0 : ((tVal - sVal) / sVal) * 100;
            const atcPct = calcPct(t.atc_rate, s.atc_rate);
            const convPct = calcPct(t.conversion, s.conversion);
            const rpsPct = calcPct(t.rps, s.rps);
            const aovPct = calcPct(t.aov, s.aov);
            const revDelta = t.revenue - s.revenue;
            
            // Overall verdict
            let winCount = 0;
            if (t.atc_rate > s.atc_rate) winCount++;
            if (t.conversion > s.conversion) winCount++;
            if (t.rps > s.rps) winCount++;
            if (t.aov > s.aov) winCount++;
            
            let verdict = '', verdictClass = '', verdictIcon = '';
            if (winCount >= 3) {{
                verdict = 'Tectonic is outperforming Shopify';
                verdictClass = 'winning';
                verdictIcon = '🏆';
            }} else if (winCount <= 1) {{
                verdict = 'Shopify is outperforming Tectonic';
                verdictClass = 'losing';
                verdictIcon = '⚠️';
            }} else {{
                verdict = 'Performance is comparable';
                verdictClass = 'neutral';
                verdictIcon = '⚖️';
            }}
            
            const period = document.getElementById('period-select').value.replace('_', ' ');
            
            document.getElementById('executive-summary').innerHTML = `
                <h4>📓 Executive Summary - ${{summaryData.clientName}} (${{period}})</h4>
                <div class="notebook-content">
                    <div class="notebook-section">
                        <h5>📊 Traffic Overview</h5>
                        <div class="notebook-grid">
                            <div class="notebook-item"><div class="nb-label">Total Sessions</div><div class="nb-value">${{formatNumber(s.sessions + t.sessions)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Shopify Sessions</div><div class="nb-value shopify">${{formatNumber(s.sessions)}} (${{s.traffic_split.toFixed(1)}}%)</div></div>
                            <div class="notebook-item"><div class="nb-label">Tectonic Sessions</div><div class="nb-value tectonic">${{formatNumber(t.sessions)}} (${{t.traffic_split.toFixed(1)}}%)</div></div>
                            <div class="notebook-item"><div class="nb-label">Total Orders</div><div class="nb-value">${{formatNumber(s.orders + t.orders)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Total Revenue</div><div class="nb-value">${{formatCurrency(s.revenue + t.revenue, currency)}}</div></div>
                        </div>
                    </div>
                    
                    <div class="notebook-divider"></div>
                    
                    <div class="notebook-section">
                        <h5>⚡ Key Metrics Comparison</h5>
                        <div class="notebook-grid">
                            <div class="notebook-item">
                                <div class="nb-label">ATC Rate</div>
                                <div class="nb-value ${{atcPct >= 0 ? 'positive' : 'negative'}}">${{atcPct >= 0 ? '+' : ''}}${{atcPct.toFixed(2)}}%</div>
                                <div class="nb-label" style="margin-top:4px;">S: ${{s.atc_rate.toFixed(2)}}% → T: ${{t.atc_rate.toFixed(2)}}%</div>
                            </div>
                            <div class="notebook-item">
                                <div class="nb-label">Conversion</div>
                                <div class="nb-value ${{convPct >= 0 ? 'positive' : 'negative'}}">${{convPct >= 0 ? '+' : ''}}${{convPct.toFixed(2)}}%</div>
                                <div class="nb-label" style="margin-top:4px;">S: ${{s.conversion.toFixed(2)}}% → T: ${{t.conversion.toFixed(2)}}%</div>
                            </div>
                            <div class="notebook-item">
                                <div class="nb-label">RPS</div>
                                <div class="nb-value ${{rpsPct >= 0 ? 'positive' : 'negative'}}">${{rpsPct >= 0 ? '+' : ''}}${{rpsPct.toFixed(2)}}%</div>
                                <div class="nb-label" style="margin-top:4px;">S: ${{formatCurrency(s.rps, currency)}} → T: ${{formatCurrency(t.rps, currency)}}</div>
                            </div>
                            <div class="notebook-item">
                                <div class="nb-label">AOV</div>
                                <div class="nb-value ${{aovPct >= 0 ? 'positive' : 'negative'}}">${{aovPct >= 0 ? '+' : ''}}${{aovPct.toFixed(2)}}%</div>
                                <div class="nb-label" style="margin-top:4px;">S: ${{formatCurrency(s.aov, currency)}} → T: ${{formatCurrency(t.aov, currency)}}</div>
                            </div>
                        </div>
                    </div>
                    
                    <div class="notebook-divider"></div>
                    
                    <div class="notebook-section">
                        <h5>💰 Revenue Impact</h5>
                        <div class="notebook-grid">
                            <div class="notebook-item"><div class="nb-label">Shopify Revenue</div><div class="nb-value shopify">${{formatCurrency(s.revenue, currency)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Tectonic Revenue</div><div class="nb-value tectonic">${{formatCurrency(t.revenue, currency)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Revenue Delta</div><div class="nb-value ${{revDelta >= 0 ? 'positive' : 'negative'}}">${{revDelta >= 0 ? '+' : ''}}${{formatCurrency(revDelta, currency)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Shopify Orders</div><div class="nb-value shopify">${{formatNumber(s.orders)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Tectonic Orders</div><div class="nb-value tectonic">${{formatNumber(t.orders)}}</div></div>
                            <div class="notebook-item"><div class="nb-label">Orders Delta</div><div class="nb-value ${{t.orders - s.orders >= 0 ? 'positive' : 'negative'}}">${{t.orders - s.orders >= 0 ? '+' : ''}}${{t.orders - s.orders}}</div></div>
                        </div>
                    </div>
                    
                    <div class="verdict-box">
                        <div class="verdict-title">${{verdictIcon}} Final Verdict <span class="performance-badge ${{verdictClass}}">${{verdict}}</span></div>
                        <div class="verdict-text">
                            Based on the ${{period}} data analysis, Tectonic ${{winCount >= 3 ? 'demonstrates superior' : winCount <= 1 ? 'shows lower' : 'shows comparable'}} performance across key metrics.
                            ${{convPct >= 0 ? '✅ Conversion is ' + convPct.toFixed(2) + '% higher' : '⚠️ Conversion is ' + Math.abs(convPct).toFixed(2) + '% lower'}} on Tectonic.
                            ${{rpsPct >= 0 ? '✅ Revenue per session improved by ' + rpsPct.toFixed(2) + '%' : '⚠️ Revenue per session decreased by ' + Math.abs(rpsPct).toFixed(2) + '%'}}.
                            ${{winCount >= 3 ? 'Recommend continuing with Tectonic rollout.' : winCount <= 1 ? 'Consider investigating performance gaps.' : 'Monitor closely for trends.'}}
                        </div>
                    </div>
                </div>
            `;
        }}
        
        function updateTrendsChart() {{
            const clientData = getClientData();
            const period = document.getElementById('period-select').value;
            const periodData = clientData.data[period];
            const currency = clientData.currency;
            let dailyData = periodData.daily_trends || [];
            const metricMap = {{ 'Sessions': {{ shopify: 'Sessions_shopify', tectonic: 'Sessions_tectonic' }}, 'Bounce Rate': {{ shopify: 'Bounce Rate_shopify', tectonic: 'Bounce Rate_tectonic' }}, 'ATC Rate': {{ shopify: 'ATC Rate_shopify', tectonic: 'ATC Rate_tectonic' }}, 'Conversion': {{ shopify: 'Conversion_shopify', tectonic: 'Conversion_tectonic' }}, 'Orders': {{ shopify: 'Orders_shopify', tectonic: 'Orders_tectonic' }}, 'Revenue': {{ shopify: 'Revenue_shopify', tectonic: 'Revenue_tectonic' }}, 'RPS': {{ shopify: 'RPS_shopify', tectonic: 'RPS_tectonic' }}, 'AOV': {{ shopify: 'AOV_shopify', tectonic: 'AOV_tectonic' }} }};
            const m = metricMap[currentTrendMetric];
            dailyData.sort((a, b) => new Date(a.Day || a.Date) - new Date(b.Day || b.Date));
            const labels = dailyData.map(d => {{ const date = new Date(d.Day || d.Date); return date.toLocaleDateString('en-IN', {{ month: 'short', day: 'numeric' }}); }});
            const shopifyData = dailyData.map(d => d[m.shopify] || 0);
            const tectonicData = dailyData.map(d => d[m.tectonic] || 0);
            if (trendsChart) trendsChart.destroy();
            trendsChart = new Chart(document.getElementById('trends-chart').getContext('2d'), {{ type: currentTrendChartType, data: {{ labels: labels, datasets: [{{ label: 'Shopify', data: shopifyData, borderColor: '#96bf48', backgroundColor: currentTrendChartType === 'bar' ? 'rgba(150, 191, 72, 0.7)' : 'rgba(150, 191, 72, 0.1)', fill: currentTrendChartType === 'line', tension: 0.4, borderRadius: 8 }}, {{ label: 'Tectonic', data: tectonicData, borderColor: '#667eea', backgroundColor: currentTrendChartType === 'bar' ? 'rgba(102, 126, 234, 0.7)' : 'rgba(102, 126, 234, 0.1)', fill: currentTrendChartType === 'line', tension: 0.4, borderRadius: 8 }}] }}, options: {{ responsive: true, maintainAspectRatio: false, plugins: {{ title: {{ display: true, text: currentTrendMetric + ' Daily Trend' }} }} }} }});
            const metricAgg = ['Sessions', 'Orders', 'Revenue'].includes(currentTrendMetric) ? 'sum' : 'avg';
            let shopifyTotal = metricAgg === 'sum' ? shopifyData.reduce((a, b) => a + b, 0) : shopifyData.filter(v => v > 0).reduce((a, b) => a + b, 0) / (shopifyData.filter(v => v > 0).length || 1);
            let tectonicTotal = metricAgg === 'sum' ? tectonicData.reduce((a, b) => a + b, 0) : tectonicData.filter(v => v > 0).reduce((a, b) => a + b, 0) / (tectonicData.filter(v => v > 0).length || 1);
            const formatVal = (v) => ['Revenue', 'RPS', 'AOV'].includes(currentTrendMetric) ? formatCurrency(v, currency) : ['Bounce Rate', 'ATC Rate', 'Conversion'].includes(currentTrendMetric) ? formatDecimal(v) + '%' : formatNumber(v);
            const delta = tectonicTotal - shopifyTotal;
            const deltaClass = currentTrendMetric === 'Bounce Rate' ? (delta < 0 ? 'delta-positive' : 'delta-negative') : (delta > 0 ? 'delta-positive' : 'delta-negative');
            document.getElementById('trends-totals').innerHTML = `<h4>📊 ${{currentTrendMetric}} Period Total (${{metricAgg === 'sum' ? 'Sum' : 'Average'}})</h4><div class="totals-grid"><div class="total-item"><div class="value" style="color: #96bf48;">${{formatVal(shopifyTotal)}}</div><div class="label">Shopify</div></div><div class="total-item"><div class="value" style="color: #667eea;">${{formatVal(tectonicTotal)}}</div><div class="label">Tectonic</div></div><div class="total-item"><div class="value ${{deltaClass}}">${{delta >= 0 ? '+' : ''}}${{formatVal(delta)}}</div><div class="label">Delta</div></div></div>`;
        }}
        
        function updateHourlyChart() {{
            const clientData = getClientData();
            const period = document.getElementById('period-select').value;
            const hourlyData = clientData.data[period].hourly;
            const currency = clientData.currency;
            const metricMap = {{ 'Sessions': {{ shopify: 'Sessions_shopify', tectonic: 'Sessions_tectonic' }}, 'Bounce Rate': {{ shopify: 'Bounce Rate_shopify', tectonic: 'Bounce Rate_tectonic' }}, 'ATC Rate': {{ shopify: 'ATC Rate_shopify', tectonic: 'ATC Rate_tectonic' }}, 'Conversion': {{ shopify: 'Conversion_shopify', tectonic: 'Conversion_tectonic' }}, 'Orders': {{ shopify: 'Orders_shopify', tectonic: 'Orders_tectonic' }}, 'Revenue': {{ shopify: 'Revenue_shopify', tectonic: 'Revenue_tectonic' }}, 'RPS': {{ shopify: 'RPS_shopify', tectonic: 'RPS_tectonic' }}, 'AOV': {{ shopify: 'AOV_shopify', tectonic: 'AOV_tectonic' }} }};
            const m = metricMap[currentHourlyMetric];
            const labels = hourlyData.map(h => String(h['Hour of day']).padStart(2, '0') + ':00');
            const shopifyData = hourlyData.map(h => h[m.shopify] || 0);
            const tectonicData = hourlyData.map(h => h[m.tectonic] || 0);
            if (hourlyChart) hourlyChart.destroy();
            hourlyChart = new Chart(document.getElementById('hourly-chart').getContext('2d'), {{ type: 'line', data: {{ labels: labels, datasets: [{{ label: 'Shopify', data: shopifyData, borderColor: '#96bf48', backgroundColor: 'rgba(150, 191, 72, 0.1)', fill: true, tension: 0.4 }}, {{ label: 'Tectonic', data: tectonicData, borderColor: '#667eea', backgroundColor: 'rgba(102, 126, 234, 0.1)', fill: true, tension: 0.4 }}] }}, options: {{ responsive: true, maintainAspectRatio: false, plugins: {{ title: {{ display: true, text: currentHourlyMetric + ' by Hour' }} }} }} }});
            const metricAgg = ['Sessions', 'Orders', 'Revenue'].includes(currentHourlyMetric) ? 'sum' : 'avg';
            let shopifyTotal = metricAgg === 'sum' ? shopifyData.reduce((a, b) => a + b, 0) : shopifyData.filter(v => v > 0).reduce((a, b) => a + b, 0) / (shopifyData.filter(v => v > 0).length || 1);
            let tectonicTotal = metricAgg === 'sum' ? tectonicData.reduce((a, b) => a + b, 0) : tectonicData.filter(v => v > 0).reduce((a, b) => a + b, 0) / (tectonicData.filter(v => v > 0).length || 1);
            const formatVal = (v) => ['Revenue', 'RPS', 'AOV'].includes(currentHourlyMetric) ? formatCurrency(v, currency) : ['Bounce Rate', 'ATC Rate', 'Conversion'].includes(currentHourlyMetric) ? formatDecimal(v) + '%' : formatNumber(v);
            const delta = tectonicTotal - shopifyTotal;
            const deltaClass = currentHourlyMetric === 'Bounce Rate' ? (delta < 0 ? 'delta-positive' : 'delta-negative') : (delta > 0 ? 'delta-positive' : 'delta-negative');
            document.getElementById('hourly-totals').innerHTML = `<h4>📊 ${{currentHourlyMetric}} Totals (${{metricAgg === 'sum' ? 'Sum' : 'Average'}})</h4><div class="totals-grid"><div class="total-item"><div class="value" style="color: #96bf48;">${{formatVal(shopifyTotal)}}</div><div class="label">Shopify</div></div><div class="total-item"><div class="value" style="color: #667eea;">${{formatVal(tectonicTotal)}}</div><div class="label">Tectonic</div></div><div class="total-item"><div class="value ${{deltaClass}}">${{delta >= 0 ? '+' : ''}}${{formatVal(delta)}}</div><div class="label">Delta</div></div></div>`;
        }}
        
        function updateLPTable(records, currency) {{ const sorted = [...records].sort((a, b) => (b[currentLpSort] || 0) - (a[currentLpSort] || 0)); const totalPages = Math.ceil(sorted.length / lpPerPage); const pageData = sorted.slice((currentLpPage - 1) * lpPerPage, currentLpPage * lpPerPage); document.querySelector('#lp-table tbody').innerHTML = pageData.map(lp => {{ const dB = (lp['Delta Bounce Rate'] || 0) <= 0 ? 'delta-positive' : 'delta-negative'; const dA = (lp['Delta ATC Rate'] || 0) >= 0 ? 'delta-positive' : 'delta-negative'; const dC = (lp['Delta Conversion'] || 0) >= 0 ? 'delta-positive' : 'delta-negative'; const dR = (lp['Delta RPS'] || 0) >= 0 ? 'delta-positive' : 'delta-negative'; const dAO = (lp['Delta AOV'] || 0) >= 0 ? 'delta-positive' : 'delta-negative'; return `<tr><td title="${{lp['Landing page path']}}">${{lp['Landing page path'].substring(0,50)}}${{lp['Landing page path'].length > 50 ? '...' : ''}}</td><td class="col-group-shopify">${{formatNumber(lp.Sessions_shopify||0)}}</td><td class="col-group-shopify">${{formatDecimal(lp['Bounce Rate_shopify']||0)}}%</td><td class="col-group-shopify">${{formatDecimal(lp['ATC Rate_shopify']||0)}}%</td><td class="col-group-shopify">${{formatDecimal(lp['Conversion_shopify']||0)}}%</td><td class="col-group-shopify">${{formatNumber(lp.Orders_shopify||0)}}</td><td class="col-group-shopify">${{currency}}${{formatNumber(lp.Revenue_shopify||0)}}</td><td class="col-group-shopify">${{currency}}${{formatDecimal(lp.RPS_shopify||0)}}</td><td class="col-group-shopify">${{currency}}${{formatDecimal(lp.AOV_shopify||0)}}</td><td class="col-group-tectonic">${{formatNumber(lp.Sessions_tectonic||0)}}</td><td class="col-group-tectonic">${{formatDecimal(lp['Bounce Rate_tectonic']||0)}}%</td><td class="col-group-tectonic">${{formatDecimal(lp['ATC Rate_tectonic']||0)}}%</td><td class="col-group-tectonic">${{formatDecimal(lp['Conversion_tectonic']||0)}}%</td><td class="col-group-tectonic">${{formatNumber(lp.Orders_tectonic||0)}}</td><td class="col-group-tectonic">${{currency}}${{formatNumber(lp.Revenue_tectonic||0)}}</td><td class="col-group-tectonic">${{currency}}${{formatDecimal(lp.RPS_tectonic||0)}}</td><td class="col-group-tectonic">${{currency}}${{formatDecimal(lp.AOV_tectonic||0)}}</td><td class="col-group-delta ${{dB}}">${{(lp['Delta Bounce Rate']||0) >= 0 ? '+' : ''}}${{formatDecimal(lp['Delta Bounce Rate']||0)}}%</td><td class="col-group-delta ${{dA}}">${{(lp['Delta ATC Rate']||0) >= 0 ? '+' : ''}}${{formatDecimal(lp['Delta ATC Rate']||0)}}%</td><td class="col-group-delta ${{dC}}">${{(lp['Delta Conversion']||0) >= 0 ? '+' : ''}}${{formatDecimal(lp['Delta Conversion']||0)}}%</td><td class="col-group-delta ${{dR}}">${{(lp['Delta RPS']||0) >= 0 ? '+' : ''}}${{formatDecimal(lp['Delta RPS']||0)}}%</td><td class="col-group-delta ${{dAO}}">${{(lp['Delta AOV']||0) >= 0 ? '+' : ''}}${{formatDecimal(lp['Delta AOV']||0)}}%</td></tr>`; }}).join(''); let pH = ''; for (let i = 1; i <= Math.min(totalPages, 10); i++) pH += `<button class="page-btn ${{i === currentLpPage ? 'active' : ''}}" onclick="goToLpPage(${{i}})">${{i}}</button>`; document.getElementById('lp-pagination').innerHTML = pH; }}
        function sortLP(c, btn) {{ currentLpSort = c; currentLpPage = 1; document.querySelectorAll('#landing-pages-tab .sort-btn').forEach(b => b.classList.remove('active')); btn.classList.add('active'); updateDashboard(); }}
        function goToLpPage(p) {{ currentLpPage = p; updateDashboard(); }}
        
        function updateUTMTable(data, currency) {{ const r = data.records || []; document.querySelector('#utm-table tbody').innerHTML = r.slice(0,20).map(r => `<tr><td style="position:relative;width:auto;background:transparent;box-shadow:none;">${{r['UTM source'] || '(direct)'}}</td><td class="col-group-shopify">${{formatNumber(r.Sessions_shopify||0)}}</td><td class="col-group-shopify">${{formatDecimal(r['Conversion_shopify']||0)}}%</td><td class="col-group-shopify">${{formatNumber(r.Orders_shopify||0)}}</td><td class="col-group-shopify">${{currency}}${{formatNumber(r.Revenue_shopify||0)}}</td><td class="col-group-tectonic">${{formatNumber(r.Sessions_tectonic||0)}}</td><td class="col-group-tectonic">${{formatDecimal(r['Conversion_tectonic']||0)}}%</td><td class="col-group-tectonic">${{formatNumber(r.Orders_tectonic||0)}}</td><td class="col-group-tectonic">${{currency}}${{formatNumber(r.Revenue_tectonic||0)}}</td><td class="col-group-delta ${{(r['Delta Conversion']||0) >= 0 ? 'delta-positive' : 'delta-negative'}}">${{(r['Delta Conversion']||0) >= 0 ? '+' : ''}}${{formatDecimal(r['Delta Conversion']||0)}}%</td><td class="col-group-delta ${{(r['Delta RPS']||0) >= 0 ? 'delta-positive' : 'delta-negative'}}">${{(r['Delta RPS']||0) >= 0 ? '+' : ''}}${{formatDecimal(r['Delta RPS']||0)}}%</td></tr>`).join(''); }}
        
        document.addEventListener('DOMContentLoaded', function() {{ loadExclusions(); document.getElementById('current-client').textContent = getClientData().display_name; updateDashboard(); }});
    </script>
</body>
</html>'''
    return html

print('✅ HTML generator loaded with all features:')
print('   - Compact delta cards with pts in brackets')
print('   - All 4 metrics insights')
print('   - Executive Summary notebook')
print('   - Performance Trends + Hourly with buttons')


## Cell 9: Save HTML Dashboard

In [ ]:
# ============================================================================
# CELL 9: SAVE HTML
# ============================================================================

print("🎨 Generating HTML dashboard...")

html_content = generate_html_dashboard(ALL_DASHBOARD_DATA)

os.makedirs('output', exist_ok=True)
html_path = 'output/rca_dashboard.html'

with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"✅ Dashboard saved to: {os.path.abspath(html_path)}")
print(f"   Clients included: {', '.join(ALL_DASHBOARD_DATA.keys())}")

display(HTML(f'''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
    <h3 style="margin: 0 0 10px 0;">🚀 Dashboard Ready!</h3>
    <p style="margin: 0;">Clients: {', '.join(ALL_DASHBOARD_DATA.keys())}</p>
    <p style="margin: 0;">Run the next cell to launch in browser</p>
</div>
'''))

## Cell 10: Launch Dashboard in Browser

In [ ]:
# ============================================================================
# CELL 10: LAUNCH IN BROWSER
# ============================================================================

import http.server
import socketserver
import threading

PORT = 8080

class Handler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory='output', **kwargs)
    def log_message(self, format, *args):
        pass

def start_server():
    try:
        with socketserver.TCPServer(("", PORT), Handler) as httpd:
            httpd.serve_forever()
    except OSError:
        pass

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

url = f"http://localhost:{PORT}/rca_dashboard.html"

print("="*60)
print(f"🌐 Server started on port {PORT}")
print(f"📊 URL: {url}")
print("="*60)

webbrowser.open(url)

display(HTML(f'''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
    <h2>✅ Dashboard is Live!</h2>
    <a href="{url}" target="_blank" style="color: white; font-size: 1.2em;">{url}</a>
</div>
'''))